# Paper B batch — C1 NER: seeds, space-joined control, byte-BPE test
Three blocks, all at the seed-42 multilingual settings (LR 3e-5 joint+BIO, O_WEIGHT 0.615, WORK_LOCAL=1).

| Block | Question | Runs | Drive root |
|---|---|---|---|
| A | Does the multilingual dose-response hold over 3 seeds? → run_08 + TOST | 4 enc × seeds 123,7 × {joint,bio} = 16 | `IdiomatorRigor_NER_multi` |
| B | Byte-level BPE prediction: RoBERTa gap stays flat, XLM-R (same arch, SP) does not | 2 enc × 3 seeds × 2 = 12, English only | `IdiomatorRigor_NER_en_bbpe` |
| C | Causal control: same English examples, punctuation detached → XLM-R gap vanishes | 2 enc × 3 seeds × 2 = 12, English only | `IdiomatorRigor_NER_en_sj` |

Runtime → **A100** (T4 works, slower). Run top to bottom. Every launch cell skips runs whose `metrics.json` already exists on Drive, so a disconnect is safe — just rerun. **After any code change, set `FORCE=1` for the affected block.**

In [ ]:
# 1. Setup: Drive + repo + deps + GPU gate
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/repo && git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r Requirements.txt
import torch
assert torch.cuda.is_available(), 'CPU-only runtime (happens after restarts) — Runtime > Change runtime type > GPU'
print('Device: cuda —', torch.cuda.get_device_name(0))

In [ ]:
# 2. Preflight: commit has Paper B changes, Drive roots writable, data has the punctuation precondition
import os, json, glob, time
assert 'SPACE_JOIN' in open('experiments/rigor/run_31_ner_data_prep.py').read(), 'old commit — push the Paper B changes first'
assert 'BBPE' in open('experiments/rigor/run_08_extended_gold.py').read(), 'old commit — push the Paper B changes first'
M = '/content/drive/MyDrive'
ROOT_A, ROOT_B, ROOT_C = f'{M}/IdiomatorRigor_NER_multi', f'{M}/IdiomatorRigor_NER_en_bbpe', f'{M}/IdiomatorRigor_NER_en_sj'
for r in (ROOT_A, ROOT_B, ROOT_C):
    os.makedirs(r, exist_ok=True)
    assert 'drive' in os.path.realpath(r).lower()
    open(f'{r}/.probe','w').write('ok'); assert open(f'{r}/.probe').read()=='ok'; os.remove(f'{r}/.probe')
P = set(",.;:!?'\")]}…।॥")
def punct_rate(path, lang=None):
    rows = [json.loads(l) for l in open(path)]
    rows = [r for r in rows if lang is None or r['language']==lang]
    return len(rows), sum(r['span_end']<len(r['sentence']) and r['sentence'][r['span_end']] in P for r in rows)/len(rows)
for lang in ['English','Spanish','Indonesian','Telugu','Hindi']:
    n, rate = punct_rate('data/ner_wikiann_multi/Splits/test.jsonl', lang)
    print(f'{lang:11} n={n}  followed-by-punct {rate:.1%}')
assert punct_rate('data/ner_wikiann_multi/Splits/test.jsonl','English')[1] > 0.10, 'not the detokenized build'

## Block A — multilingual seeds 123 and 7

In [ ]:
# 3. Seed-42 runs must exist (same root) — the new seeds are only useful next to them.
#    Also prints the BIO O-weight from the s42 logs and the XLM-R BIO epoch losses (epoch-4 spike check).
layout = {'rembert':'rembert','xlmr':'flip','mbert':'flip','muril':'flip'}
for enc, sub in layout.items():
    for sysn in ('joint','bio'):
        d = f'{ROOT_A}/{sub}/{enc}_{sysn}_s42'
        ok = os.path.getsize(f'{d}/test_predictions.jsonl')>0 if os.path.exists(f'{d}/test_predictions.jsonl') else False
        print(('✓' if ok else '✗ MISSING'), d)
!grep -h "O weight" {ROOT_A}/*/*_bio_s42/console.log
!grep -i "epoch.*loss" {ROOT_A}/flip/xlmr_bio_s42/console.log | head -12

In [ ]:
# 4. Launch seeds 123 + 7, all 4 encoders (16 runs). Resumable.
FORCE = 0
ENV_A = (f"DRIVE_OUT={ROOT_A} DATA_DIR=data/ner_wikiann_multi/Splits LANGS='English Spanish Hindi Telugu' "
         f"TEST_LANGS='English Spanish Hindi Telugu Indonesian' O_WEIGHT=0.615 LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 FORCE={FORCE}")
ENCODERS = [('google/rembert','rembert','rembert'), ('xlm-roberta-base','xlmr','flip'),
            ('bert-base-multilingual-cased','mbert','flip'), ('google/muril-base-cased','muril','flip')]
for model, short, sub in ENCODERS:
    !{ENV_A} SEEDS="123 7" OUT_SUB={sub} MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 5. Read back from Drive (rows + mtime), then score: run_08 (3 modes), run_08b (SP strip CI), run_13 (TOST vs WP)
def readback(root):
    for f in sorted(glob.glob(f'{root}/*/*_s*/test_predictions.jsonl')):
        cfg = f.replace('test_predictions.jsonl','config.json')
        lr = json.load(open(cfg)).get('lr') if os.path.exists(cfg) else 'n/a (pre-config-persist run)'
        print(f'{sum(1 for _ in open(f)):5d} rows  lr={lr}  {time.ctime(os.path.getmtime(f))}  {f[len(root)+1:]}')
readback(ROOT_A)
!python experiments/rigor/run_08_extended_gold.py --preds-root {ROOT_A} --seeds 42 123 7 --out {ROOT_A}/run_08_ner_multi_3seed.json --force
!python experiments/rigor/run_08b_strip_ci.py --preds-root {ROOT_A} --seeds 42 123 7 | tee {ROOT_A}/run_08b_ner_multi_3seed.txt
!python experiments/rigor/run_13_tost_equivalence.py --preds-root {ROOT_A} --seeds 42 123 7 --out {ROOT_A}/run_13_ner_multi_3seed_wp.json --force
!python experiments/rigor/run_13_tost_equivalence.py --preds-root {ROOT_A} --seeds 42 123 7 --reference zero --out {ROOT_A}/run_13_ner_multi_3seed_zero.json --force

## Blocks B + C — English only
B = detokenized English (from the multilingual split). C = the same English examples rebuilt with `--space-join`.
XLM-R and RoBERTa share the architecture; they differ in tokenizer (SentencePiece vs byte-level BPE) and pretraining data.

In [ ]:
# 6. Build the space-joined English split and prove it is the same examples
!pip install -q datasets
!python experiments/rigor/run_31_ner_data_prep.py --lang en --language-name English --out-dir data/ner_wikiann_en_sj/Splits --space-join --force
norm = lambda s: ''.join(s.split())
for split in ('train','dev','test'):
    sj  = [json.loads(l) for l in open(f'data/ner_wikiann_en_sj/Splits/{split}.jsonl')]
    det = [r for r in map(json.loads, open(f'data/ner_wikiann_multi/Splits/{split}.jsonl')) if r['language']=='English']
    assert len(sj)==len(det), (split, len(sj), len(det))
    assert all(norm(a['sentence'])==norm(b['sentence']) and norm(a['sentence'][a['span_start']:a['span_end']])==norm(b['sentence'][b['span_start']:b['span_end']])
               for a,b in zip(sj,det)), f'{split}: example mismatch'
    print(f'✓ {split}: {len(sj)} identical examples')
n, rate = punct_rate('data/ner_wikiann_en_sj/Splits/test.jsonl'); print(f'space-joined test followed-by-punct {rate:.1%}')
assert rate == 0
!mkdir -p {ROOT_C}/data && cp -r data/ner_wikiann_en_sj/Splits {ROOT_C}/data/   # provenance copy

In [ ]:
# 7. Mechanism check, no GPU: share of English test gold spans whose boundaries are NOT token boundaries.
#    Descriptive only. Prediction if byte-BPE behaves like WordPiece: RoBERTa ≈ mBERT, both well below XLM-R.
from transformers import AutoTokenizer
rows = [r for r in map(json.loads, open('data/ner_wikiann_multi/Splits/test.jsonl')) if r['language']=='English']
for name in ['xlm-roberta-base','google/rembert','roberta-base','bert-base-multilingual-cased']:
    tok = AutoTokenizer.from_pretrained(name)
    bad = 0
    for r in rows:
        offs = [o for o in tok(r['sentence'], return_offsets_mapping=True)['offset_mapping'] if o[1]>o[0]]
        starts, ends = {o[0] for o in offs}, {o[1] for o in offs}
        bad += not (r['span_start'] in starts and r['span_end'] in ends)
    print(f'{name:32} misaligned gold boundaries: {bad/len(rows):.1%}')

In [ ]:
# 8. RoBERTa compatibility dry-run (1 epoch, both trainers) before the full block
ENV_EN = "LANGS=English TEST_LANGS=English O_WEIGHT=0.615 LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1"
!DRIVE_OUT={ROOT_B} DATA_DIR=data/ner_wikiann_multi/Splits {ENV_EN} OUT_SUB=bbpe MODEL=roberta-base ENC_SHORT=roberta bash experiments/rigor/run_07_mdeberta_sp_replication.sh --dry-run
!rm -rf {ROOT_B}/bbpe/_dryrun_joint {ROOT_B}/bbpe/_dryrun_bio

In [ ]:
# 9. Launch B (detokenized) and C (space-joined): XLM-R + RoBERTa × seeds 42 123 7 (24 runs). Resumable.
FORCE = 0
for root, data in [(ROOT_B,'data/ner_wikiann_multi/Splits'), (ROOT_C,'data/ner_wikiann_en_sj/Splits')]:
    for model, short, sub in [('xlm-roberta-base','xlmr','flip'), ('roberta-base','roberta','bbpe')]:
        !DRIVE_OUT={root} DATA_DIR={data} {ENV_EN} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB={sub} MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 10. Read back + score both roots. Predictions: B xlmr original gap > 0, strip ≈ 0; B roberta ≈ 0 in every mode; C both ≈ 0.
for root, tag in [(ROOT_B,'en_bbpe'), (ROOT_C,'en_sj')]:
    readback(root)
    !BBPE=1 python experiments/rigor/run_08_extended_gold.py --preds-root {root} --seeds 42 123 7 --out {root}/run_08_ner_{tag}_3seed.json --force

## Block D — ABSA-14 aspect terms (second natural-text task)
Raw text + char offsets from HF `alexcadillon/SemEval2014Task4` (no XML needed). Single-aspect sentences only; dev is carved from train. ~33-35% of aspect terms are followed by punctuation (top of the dose-response). 4 encoders × 3 seeds × 2 systems × 2 domains = 48 small runs. Needs cells 1-2 (setup) run first in the same tab; outputs go to `IdiomatorRigor_NER_absa_{restaurants,laptops}`.

In [ ]:
# 11. Build ABSA data (public HF rows via datasets-server, stdlib) and check the precondition + size
import json, os, subprocess
for dom in ('restaurants','laptops'):
    out = f'data/absa14_{dom}/Splits'
    r = subprocess.run(['python','experiments/rigor/run_33_absa_data_prep.py','--config',dom,'--out-dir',out,'--force'],capture_output=True,text=True)
    print(f'##### {dom}'); print(r.stdout[-1500:], r.stderr[-1500:])
    meta = json.load(open(f'{out}/meta.json'))
    assert meta['splits']['test']['punct_follow_pct'] > 20, 'precondition missing — wrong (pre-tokenized?) data'
    print('n train/dev/test:', [meta['splits'][s]['n'] for s in ('train','dev','test')])
    os.makedirs(f'{M}/IdiomatorRigor_NER_absa_{dom}/data', exist_ok=True)
    !cp -r {out} {M}/IdiomatorRigor_NER_absa_{dom}/data/    # provenance copy

In [ ]:
# 12. Launch: all 4 encoders × seeds 42 123 7 per domain. O_WEIGHT comes from each domain's meta.json. Resumable.
FORCE = 0
ENCODERS = [('google/rembert','rembert','rembert'), ('xlm-roberta-base','xlmr','flip'),
            ('bert-base-multilingual-cased','mbert','flip'), ('google/muril-base-cased','muril','flip')]
for dom in ('restaurants','laptops'):
    root = f'{M}/IdiomatorRigor_NER_absa_{dom}'
    ow = json.load(open(f'data/absa14_{dom}/Splits/meta.json'))['o_weight']
    env = (f"DRIVE_OUT={root} DATA_DIR=data/absa14_{dom}/Splits LANGS=English TEST_LANGS=English "
           f"O_WEIGHT={ow} LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 FORCE={FORCE} SEEDS='42 123 7'")
    for model, short, sub in ENCODERS:
        !{env} OUT_SUB={sub} MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 13. Read back (rows must be equal joint vs bio — a missing BIO row is the SentencePiece skip) then score
import glob, time, subprocess
for dom in ('restaurants','laptops'):
    root = f'{M}/IdiomatorRigor_NER_absa_{dom}'
    files = sorted(glob.glob(f'{root}/*/*_s*/test_predictions.jsonl'))
    print(f'\n##### {dom}: {len(files)} prediction files (expect 24)')
    for f in files:
        cfg = json.load(open(f.replace('test_predictions.jsonl','config.json'))) if os.path.exists(f.replace('test_predictions.jsonl','config.json')) else {}
        print(f"{sum(1 for _ in open(f)):5d} rows  lr={cfg.get('lr')}  {time.ctime(os.path.getmtime(f))}  {f[len(root)+1:]}")
    for cmd in (['experiments/rigor/run_08_extended_gold.py','--preds-root',root,'--seeds','42','123','7','--out',f'{root}/run_08_absa_{dom}_3seed.json','--force'],
                ['experiments/rigor/run_13_tost_equivalence.py','--preds-root',root,'--seeds','42','123','7','--out',f'{root}/run_13_absa_{dom}_wp.json','--force']):
        r = subprocess.run(['python']+cmd,capture_output=True,text=True); print(r.stdout[-5000:], r.stderr[-1500:])